<a href="https://colab.research.google.com/github/saparbayev-azizbek-12/bi-and-ai-talents-dl/blob/main/lesson-27/RNNCell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F

class RNNCell(nn.Module):
    def __init__(self, input_size, hidden_size, bias=True, nonlinearity='tanh'):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.bias = bias
        if nonlinearity not in ['relu', 'tanh']:
            raise ValueError(f"Nonelinearity must be 'tanh' or 'relu' not {nonlinearity}")
        self.nonlinearity = nonlinearity

        self.Wih = nn.Parameter(torch.empty(hidden_size, input_size))
        self.Whh = nn.Parameter(torch.empty(hidden_size, hidden_size))

        if bias:
            self.bih = nn.Parameter(torch.empty(hidden_size))
            self.bhh = nn.Parameter(torch.empty(hidden_size))
        else:
            self.bih = None
            self.bhh = None

    def __repr__(self):
        return f"input_size={self.input_size}, hidden_size={self.hidden_size}"

    def reset_parameters(self):
        k = math.sqrt(1.0 / self.hidden_size)
        for w in self.parameters():
            return nn.init.uniform_(w, -k, k)

    def forward(self, x, hx=None):
        batch_size, _ = x.shape

        if hx is None:
            hx = torch.zeros(batch_size, self.hidden_size)

        pre = F.linear(x, self.Wih, self.bih) + F.linear(hx, self.Whh, self.bhh)
        post = torch.tanh(pre) if self.nonlinearity == 'tanh' else torch.relu(pre)
        return post


In [ ]:
rnn = RNNCell(10, 20)
input = torch.randn(3, 10)
hx = torch.randn(3, 20)
output = []
hx = rnn(input, hx)
output.append(hx)

In [ ]:
class RNN(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers=1,
        nonlinearity='tanh',
        bias=True,
        batch_first=False,
        dropout=0.0,
        bidirectional=False,
    ):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.nonlinearity = nonlinearity
        self.bias = bias
        self.batch_first = batch_first
        self.dropout = dropout
        self.bidirectional = bidirectional
        self.num_directions = 2 if self.bidirectional else 1

        self.cells = nn.ModuleList()
        for layer in range(self.num_layers):
            in_size = self.input_size if layer == 0 else self.hidden_size
            for i in range(self.num_directions):
                self.cells.append(RNNCell(in_size, self.hidden_size))

    def _cell(self, layer, direction):
        return self.cells[layer * self.num_directions + direction]

    def forward(self, x, hx=None):
        if self.batch_first:
            x = x.transpose(0, 1)
        seq_len, batch_size, _ = x.shape

        if hx is None:
            hx = torch.zeros(
                self.num_layers,
                batch_size, self.hidden_size, device=x.device
            )

        current_input = x
        h_n_list = []
        for layer in range(self.num_layers):
            output_per_direction = []
            for direction in range(self.num_directions):
                cell = self._cell(layer, direction)
                h_t = hx[layer * self.num_directions + direction]

                indices = range(seq_len) if direction == 0 else range(seq_len - 1, -1, -1)
                step_outputs = []
                for t in indices:
                    h_t = cell(current_input[t], h_t)
                    step_outputs.append(h_t)

                if direction == 1:
                    step_outputs.reverse()

                step_outputs = torch.stack(step_outputs, dim=0)
                output_per_direction.append(step_outputs)
                h_n_list.append(h_t)

            current_input = torch.cat(output_per_direction, dim=-1)
            h_n = torch.stack(h_n_list, dim=0)

            if self.dropout > 0 and layer != self.num_layers - 1 and self.training:
                current_input = F.dropout(current_input, p=self.dropout, training=self.training)

            if self.batch_first:
                current_input = current_input.transpose(0, 1)
        return current_input, h_n

In [ ]:
rnn = RNN(10, 20, 2)
input = torch.randn(5, 3, 10)
h0 = torch.randn(2, 3, 20)
output, hn = rnn(input, h0)

In [ ]:
output.shape

torch.Size([5, 3, 20])